# Lesson 4: ReAct Reasoning Loop — Multi-Step Analysis

## What You'll Learn

1. **ReAct pattern** — Reason + Act in an interleaved loop (the backbone of modern agents)
2. **Multi-step planning** — Agent decomposes complex questions into sub-queries
3. **Iteration control** — Max steps, cost budgets, and loop termination
4. **Tool selection** — Agent picks the right tool (SQL vs Python vs schema) per step
5. **Structured plans** — Agent outputs an explicit plan before executing

---

## The ReAct Pattern — Why It Matters

### Before ReAct: Single-shot agents
```
User question → LLM → Answer (one shot, often wrong for complex questions)
```

### After ReAct: Iterative reasoning
```
User question → LLM THINKS → ACTs (tool call) → OBSERVES result
                    ↑                                    |
                    └────────────────────────────────────┘
                         (loop until answer is complete)
```

**ReAct** (Yao et al., 2022) interleaves reasoning traces with actions. The key insight:
LLMs that *think out loud* about what to do next make fewer mistakes than those that
jump straight to an answer.

### How PydanticAI implements ReAct

PydanticAI's agent loop IS a ReAct loop. When you give an agent tools:

1. **LLM receives**: system prompt + user message + tool schemas
2. **LLM decides**: call a tool OR return final output
3. **If tool call**: PydanticAI executes it, feeds result back to LLM
4. **LLM decides again**: call another tool OR return final output
5. **Repeat** until the LLM returns structured output (not a tool call)

The agent naturally does multi-step reasoning when the system prompt encourages it.
But for *complex* analysis, we can make this explicit with a **planning step**.

---

## Setup

In [1]:
import os
import re
import sys
import time
from pathlib import Path
from dotenv import load_dotenv
import nest_asyncio
nest_asyncio.apply()


for _env_candidate in (Path('.env'), Path('../.env')):
    if _env_candidate.exists():
        load_dotenv(_env_candidate)
        break

# LM Studio OpenAI-compatible local setup
if os.getenv("LMSTUDIO_BASE_URL") and not os.getenv("OPENAI_BASE_URL"):
    os.environ["OPENAI_BASE_URL"] = os.getenv("LMSTUDIO_BASE_URL").rstrip("/") + "/v1"
if os.getenv("LMSTUDIO_BASE_URL") and not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = "lm-studio"
sys.path.insert(0, str(Path("..").resolve() / "src"))

import pandas as pd
import duckdb
from dataclasses import dataclass, field
from pydantic import BaseModel, Field, field_validator
from pydantic_ai import Agent, RunContext, ModelRetry
from pydantic_ai.usage import UsageLimits
from pydantic_ai.exceptions import UsageLimitExceeded, UnexpectedModelBehavior, ModelHTTPError

from analyst.tools.code_executor import execute_code_subprocess
from analyst.tools.schema_inspector import inspect_schema

DATA_DIR = str(Path("../data").resolve())
sales_df = pd.read_csv(f"{DATA_DIR}/sample_sales.csv")
employees_df = pd.read_csv(f"{DATA_DIR}/sample_employees.csv")

print(f"Loaded: sales ({sales_df.shape}), employees ({employees_df.shape})")

def run_agent_with_limits(
    agent: Agent,
    prompt: str,
    *,
    deps=None,
    request_limit: int = 80,
    tool_calls_limit: int = 60,
):
    """Run an agent with explicit request/tool-call budgets for notebook safety."""
    limits = UsageLimits(
        request_limit=request_limit,
        tool_calls_limit=tool_calls_limit,
    )
    try:
        if deps is None:
            return agent.run_sync(prompt, usage_limits=limits)
        return agent.run_sync(prompt, deps=deps, usage_limits=limits)
    except UsageLimitExceeded as exc:
        print(f"Stopped by usage limits: {exc}")
        if deps is not None and hasattr(deps, "step_log") and getattr(deps, "step_log"):
            print("Recent steps before stop:")
            for step in getattr(deps, "step_log")[-5:]:
                print(f"  - {step}")
        raise
    except ModelHTTPError as exc:
        message = str(exc)
        if "No user query found in messages" in message:
            raise RuntimeError(
                "LM Studio prompt template is incompatible with this multi-step tool-calling flow.\n"
                "Fix in LM Studio:\n"
                "1) Use an lmstudio-community model/template for your model family.\n"
                "2) In model settings -> Prompt Template, remove any block that raises when no direct user turn is present.\n"
                "3) Disable model 'thinking' output for API calls when using strict structured outputs.\n"
                "4) Reload model and rerun this cell."
            ) from exc
        raise
    except UnexpectedModelBehavior as exc:
        message = str(exc)
        if "output validation" in message.lower():
            print("Model returned non-conforming structured output after retries.")
            print("This is common with local models on long ReAct runs.")
            print("Try: simpler prompt, higher retries, and disabling thinking output.")
            if deps is not None and hasattr(deps, "step_log") and getattr(deps, "step_log"):
                print("Recent steps before validation failure:")
                for step in getattr(deps, "step_log")[-5:]:
                    print(f"  - {step}")
        raise


Loaded: sales ((40, 8)), employees ((20, 8))


---

## Part 1: Implicit ReAct — PydanticAI's Built-in Loop

Let's first see how PydanticAI naturally does multi-step reasoning
when given multiple tools and a complex question.

In [2]:
@dataclass
class AnalystDeps:
    tables: dict[str, pd.DataFrame] = field(default_factory=dict)
    data_dir: str = ""
    step_log: list[dict] = field(default_factory=list)


class AnalysisResult(BaseModel):
    answer: str = Field(default="", description="Complete answer with specific numbers")
    method: str = Field(default="", description="Tools and approach used")
    key_findings: list[str] = Field(default_factory=list, description="Bullet-point findings")
    steps_taken: int = Field(default=0, description="Number of tool calls made")
    confidence: float = Field(default=0.7, description="Confidence between 0 and 1")

    @field_validator("key_findings", mode="before")
    @classmethod
    def coerce_key_findings(cls, v):
        if v is None:
            return []
        if isinstance(v, str):
            items = [line.strip("- *•	 ") for line in v.splitlines() if line.strip()]
            return items or [v.strip()]
        if isinstance(v, list):
            return [str(item) for item in v]
        return [str(v)]

    @field_validator("steps_taken", mode="before")
    @classmethod
    def coerce_steps_taken(cls, v):
        if isinstance(v, int):
            return max(v, 0)
        if isinstance(v, float):
            return max(int(v), 0)
        if isinstance(v, str):
            match = re.search(r"-?\d+", v)
            if match:
                return max(int(match.group()), 0)
        return 0

    @field_validator("confidence", mode="before")
    @classmethod
    def coerce_confidence(cls, v):
        if isinstance(v, str):
            raw = v.strip().replace("%", "")
            try:
                v = float(raw)
            except ValueError:
                return 0.7
        try:
            conf = float(v)
        except (TypeError, ValueError):
            return 0.7
        if conf > 1.0 and conf <= 100.0:
            conf = conf / 100.0
        return max(0.0, min(conf, 1.0))


react_agent = Agent(
    "openai:local-model",
    deps_type=AnalystDeps,
    output_type=AnalysisResult,
    system_prompt=(
        "You are a senior data analyst. Think step-by-step.\n\n"
        "STRATEGY (ReAct):\n"
        "1. THINK: What do I need to find out first?\n"
        "2. ACT: Use the most appropriate tool\n"
        "3. OBSERVE: Read the result carefully\n"
        "4. THINK: Do I have enough to answer? If not, what next?\n"
        "5. Repeat until confident, then give your final structured answer.\n\n"
        "For complex questions, break them into smaller sub-queries.\n"
        "Always inspect schema before querying unfamiliar data."
    ),
    retries=3,
)


@react_agent.system_prompt
def table_context(ctx: RunContext[AnalystDeps]) -> str:
    lines = ["\nAvailable tables:"]
    for name, df in ctx.deps.tables.items():
        cols = ", ".join(f"{c} ({df[c].dtype})" for c in df.columns)
        lines.append(f"  '{name}': {len(df)} rows | {cols}")
    return "\n".join(lines)


@react_agent.tool
def inspect_table(ctx: RunContext[AnalystDeps], table_name: str) -> str:
    """Get detailed schema: columns, types, stats, sample values."""
    if table_name not in ctx.deps.tables:
        return f"Not found. Available: {list(ctx.deps.tables.keys())}"
    ctx.deps.step_log.append({"tool": "inspect_table", "table": table_name})
    return inspect_schema(ctx.deps.tables[table_name], table_name)


@react_agent.tool
def query_sql(ctx: RunContext[AnalystDeps], sql: str) -> str:
    """Run SQL query against available tables using DuckDB."""
    ctx.deps.step_log.append({"tool": "query_sql", "sql": sql})
    conn = duckdb.connect()
    try:
        for name, df in ctx.deps.tables.items():
            conn.register(name, df)
        result = conn.execute(sql).fetchdf()
        if len(result) > 50:
            return f"First 50 of {len(result)} rows:\n{result.head(50).to_string()}"
        return result.to_string()
    except Exception as e:
        raise ModelRetry(f"SQL error: {e}\nQuery: {sql}")
    finally:
        conn.close()


@react_agent.tool
def run_python(ctx: RunContext[AnalystDeps], code: str) -> str:
    """Execute Python code in sandbox. pandas/numpy/matplotlib available.
    Use DATA_DIR for file paths. Always print() results."""
    ctx.deps.step_log.append({"tool": "run_python", "code_len": len(code)})
    result = execute_code_subprocess(
        code=code, data_dir=ctx.deps.data_dir, timeout_seconds=30,
    )
    if not result.success:
        raise ModelRetry(f"Code failed:\n{result.stderr}")
    parts = []
    if result.stdout:
        parts.append(result.stdout[:3000])
    if result.generated_files:
        parts.append(f"Charts: {result.generated_files}")
    return "\n".join(parts) or "Done (no output)."


print("ReAct agent ready with 3 tools: inspect_table, query_sql, run_python")


ReAct agent ready with 3 tools: inspect_table, query_sql, run_python


In [3]:
# -----------------------------------------------------------------
# Complex question that requires multiple steps
# -----------------------------------------------------------------

deps = AnalystDeps(
    tables={"sales": sales_df, "employees": employees_df},
    data_dir=DATA_DIR,
)

result = run_agent_with_limits(
    react_agent,
    "Compare Electronics vs Home categories across all metrics: "
    "total revenue, total profit, average profit margin, "
    "number of unique products, and best-selling product in each. "
    "Which category is the better business?",
    deps=deps,
    request_limit=120,
    tool_calls_limit=80,
)

print(f"Answer:\n{result.output.answer}\n")
print(f"Method: {result.output.method}")
print(f"\nKey findings:")
for f in result.output.key_findings:
    print(f"  - {f}")
print(f"\nSteps taken: {result.output.steps_taken}")
print(f"Confidence: {result.output.confidence:.0%}")
print(f"API requests: {result.usage().requests}")
print(f"\nStep log:")
for i, step in enumerate(deps.step_log, 1):
    print(f"  {i}. {step['tool']}: {str(step)[:100]}")


Answer:
**Electronics is the better business category.**

### Complete Metrics Comparison:

| Metric | Electronics | Home | Winner |
|--------|-------------|------|---------|
| Total Revenue | $86,322.50 | $61,372.60 | Electronics (+40.7%) |
| Total Profit | $34,529.00 | $24,549.04 | Electronics (+40.7%) |
| Average Profit Margin | 40.0% | 40.0% | Tie |
| Unique Products | 3 | 3 | Tie |
| Total Units Sold | 2,750 | 2,740 | Electronics (+0.36%) |
| Avg Revenue Per Unit | $31.39 | $22.40 | Electronics (+40.1%) |

### Best-Selling Products:
- **Electronics**: Widget A - 1,230 units, $36,887.70 revenue
- **Home**: Gadget X - 1,235 units, $18,512.65 revenue

### Key Findings:
• Electronics generates 40.7% more revenue and profit despite similar unit volumes
• Both categories maintain identical 40% profit margins
• Electronics products command 40% higher average revenue per unit ($31.39 vs $22.40)
• Electronics' best-seller generates 99% more revenue than Home's best-seller
• Both categories

### Observing the ReAct loop

Notice how the agent:
1. Inspected the schema first (THINK: what columns exist?)
2. Ran SQL for each metric (ACT: get the data)
3. Compared results (OBSERVE + THINK: which is better?)
4. Synthesized a final answer (RESPOND)

This is **implicit** ReAct — PydanticAI's tool loop naturally creates it.

---

## Part 2: Explicit Planning — Plan-then-Execute

For even more complex questions, we can add an **explicit planning step**.
The agent first outputs a structured plan, then executes each step.

This is the **Plan-and-Solve** pattern (Wang et al., 2023).

### Why plan explicitly?

| Approach | Pros | Cons |
|----------|------|------|
| Implicit (let LLM loop) | Simple, flexible | Can wander, hard to debug |
| Explicit plan first | Debuggable, predictable, auditable | Extra LLM call, plan might be wrong |

Production agents often use **explicit planning** because:
- You can **log** and **review** the plan before execution
- You can **estimate cost** before committing (count planned tool calls)
- Users can **approve or modify** the plan (human-in-the-loop)

### How does the planner generate structured steps?

The planner agent uses `output_type=AnalysisPlan` — the same structured output
pattern from Lesson 1. PydanticAI sends the Pydantic model's JSON schema to the LLM,
and the LLM generates a response that conforms to it:

```python
class AnalysisPlan(BaseModel):
    question: str           # Restated question
    steps: list[AnalysisStep]  # Each step has: description, tool, query_hint
    estimated_complexity: int   # 1-10

# The LLM sees this schema and generates a matching JSON object.
# PydanticAI validates it → you get a typed Python object.
```

The planner LLM call does NOT execute any tools. It only thinks about
what tools to call and in what order. Execution happens in a separate step.

In [4]:
# -----------------------------------------------------------------
# Step 1: The Planner Agent — produces a structured plan
# -----------------------------------------------------------------

class AnalysisStep(BaseModel):
    step_number: int = 1
    description: str = Field(default="", description="What this step does")
    tool: str = Field(default="sql", description="Which tool to use: 'sql', 'python', or 'inspect'")
    query_hint: str = Field(default="", description="SQL query or code approach for this step")

    @field_validator("step_number", mode="before")
    @classmethod
    def coerce_step_number(cls, v):
        if isinstance(v, int):
            return max(1, v)
        if isinstance(v, float):
            return max(1, int(v))
        if isinstance(v, str):
            match = re.search(r"\d+", v)
            if match:
                return max(1, int(match.group()))
        return 1

    @field_validator("tool", mode="before")
    @classmethod
    def coerce_tool(cls, v):
        allowed = {"sql", "python", "inspect"}
        tool = str(v).strip().lower() if v is not None else "sql"
        return tool if tool in allowed else "sql"


class AnalysisPlan(BaseModel):
    question: str = Field(default="", description="Original question, restated for clarity")
    steps: list[AnalysisStep] = Field(default_factory=list, description="Ordered analysis steps")
    estimated_complexity: int = Field(default=5, description="Estimated complexity 1-10")
    estimated_tool_calls: int = Field(default=3, description="How many tool calls this will need")

    @field_validator("estimated_complexity", mode="before")
    @classmethod
    def coerce_complexity(cls, v):
        try:
            iv = int(float(v))
        except (TypeError, ValueError):
            iv = 5
        return max(1, min(iv, 10))

    @field_validator("estimated_tool_calls", mode="before")
    @classmethod
    def coerce_tool_calls(cls, v):
        try:
            iv = int(float(v))
        except (TypeError, ValueError):
            iv = 3
        return max(1, iv)


planner = Agent(
    "openai:local-model",
    deps_type=AnalystDeps,
    output_type=AnalysisPlan,
    system_prompt=(
        "You are a data analysis planner. Given a question and available data, "
        "create a step-by-step plan for answering it.\n\n"
        "RULES:\n"
        "- Each step should be a single, clear action\n"
        "- Choose the right tool: 'inspect' for schema, 'sql' for queries, 'python' for complex analysis\n"
        "- Start with schema inspection if the data structure is unclear\n"
        "- SQL steps must be executable as-is in DuckDB (self-contained)\n"
        "- When using date_trunc/date functions on string dates, cast first (TRY_CAST(date AS DATE))\n"
        "- Prefer SQL for trend calculations; use Python only when SQL is clearly insufficient\n"
        "- If a step depends on prior SQL output, reference step_<n> or last_result tables\n"
        "- Keep plans concise: 2-5 steps for most questions\n"
        "- Do NOT execute anything — only plan"
    ),
)


@planner.system_prompt
def planner_context(ctx: RunContext[AnalystDeps]) -> str:
    lines = ["\nAvailable tables:"]
    for name, df in ctx.deps.tables.items():
        cols = ", ".join(f"{c} ({df[c].dtype})" for c in df.columns)
        lines.append(f"  '{name}': {len(df)} rows | {cols}")
    return "\n".join(lines)


# Test the planner
plan_result = run_agent_with_limits(
    planner,
    "Find the most profitable product-region combination and explain "
    "what makes it stand out compared to others.",
    deps=deps,
    request_limit=60,
    tool_calls_limit=30,
)

plan = plan_result.output
print(f"Question: {plan.question}")
print(f"Complexity: {plan.estimated_complexity}/10")
print(f"Estimated tool calls: {plan.estimated_tool_calls}")
print(f"\nPlan:")
for step in plan.steps:
    print(f"  {step.step_number}. [{step.tool}] {step.description}")
    print(f"     Hint: {step.query_hint[:100]}")


Question: Find the most profitable product-region combination and explain what makes it stand out compared to others.
Complexity: 4/10
Estimated tool calls: 3

Plan:
  1. [inspect] Inspect the sales table schema to confirm available columns for calculating profitability (revenue and cost).
     Hint: DESCRIBE sales
  2. [sql] Calculate total profit (revenue - cost) for each unique product-region combination and identify the top performer.
     Hint: SELECT product, region, SUM(revenue - cost) as total_profit FROM sales GROUP BY product, region ORDE
  3. [sql] Calculate aggregate statistics (total profit, total revenue, average unit price) for the top product-region combination and compare against all other combinations to highlight what makes it stand out.
     Hint: WITH top_combo AS (SELECT product, region FROM sales GROUP BY product, region ORDER BY SUM(revenue -


In [5]:
# -----------------------------------------------------------------
# Step 2: The Executor — runs the plan step by step
# -----------------------------------------------------------------

class StepResult(BaseModel):
    step_number: int
    tool_used: str
    output: str
    success: bool


class FinalAnalysis(BaseModel):
    answer: str = Field(default="", description="Complete answer with specific numbers")
    key_findings: list[str] = Field(default_factory=list)
    confidence: float = Field(default=0.7, description="Confidence between 0 and 1")

    @field_validator("key_findings", mode="before")
    @classmethod
    def coerce_key_findings(cls, v):
        if v is None:
            return []
        if isinstance(v, str):
            items = [line.strip("- *•	 ") for line in v.splitlines() if line.strip()]
            return items or [v.strip()]
        if isinstance(v, list):
            return [str(item) for item in v]
        return [str(v)]

    @field_validator("confidence", mode="before")
    @classmethod
    def coerce_confidence(cls, v):
        if isinstance(v, str):
            raw = v.strip().replace("%", "")
            try:
                v = float(raw)
            except ValueError:
                return 0.7
        try:
            conf = float(v)
        except (TypeError, ValueError):
            return 0.7
        if conf > 1.0 and conf <= 100.0:
            conf = conf / 100.0
        return max(0.0, min(conf, 1.0))


def execute_plan(plan: AnalysisPlan, deps: AnalystDeps) -> list[StepResult]:
    """Execute each step and persist SQL intermediates for downstream steps.

    SQL state:
    - Base tables are registered once at the start.
    - Each successful SQL step is registered as step_<n> and last_result.
    - If a step references derived columns not present in base tables,
      we attempt a light recovery by rerunning against last_result.
    """
    results: list[StepResult] = []
    conn = duckdb.connect()
    has_last_result = False
    last_sql_df = None

    try:
        for name, df in deps.tables.items():
            conn.register(name, df)

        for step in plan.steps:
            print(f"  Executing step {step.step_number}: {step.description[:60]}...")

            try:
                if step.tool == "inspect":
                    table = next(
                        (t for t in deps.tables if t in step.query_hint.lower()),
                        list(deps.tables.keys())[0],
                    )
                    output = inspect_schema(deps.tables[table], table)

                elif step.tool == "sql":
                    query = step.query_hint.strip()
                    df_out = conn.execute(query).fetchdf()
                    last_sql_df = df_out

                    conn.register(f"step_{step.step_number}", df_out)
                    conn.register("last_result", df_out)
                    has_last_result = True

                    output = df_out.to_string(index=False)

                elif step.tool == "python":
                    python_code = step.query_hint
                    if last_sql_df is not None:
                        bridge_csv = Path(deps.data_dir) / "_last_result_step.csv"
                        last_sql_df.to_csv(bridge_csv, index=False)
                        python_code = (
                            "import pandas as pd\n"
                            + f"last_result = pd.read_csv(r'{bridge_csv}')\n"
                            + "df = last_result.copy()\n\n"
                            + step.query_hint
                        )

                    exec_result = execute_code_subprocess(
                        code=python_code,
                        data_dir=deps.data_dir,
                        timeout_seconds=30,
                    )
                    output = exec_result.stdout if exec_result.success else f"Error: {exec_result.stderr}"
                else:
                    output = f"Unknown tool: {step.tool}"

                results.append(StepResult(
                    step_number=step.step_number,
                    tool_used=step.tool,
                    output=output[:2000],
                    success=True,
                ))

            except Exception as e:
                recovered = False

                if step.tool == "sql":
                    msg = str(e)
                    lower = msg.lower()

                    # Recovery 1: common DuckDB date_trunc binder issue on string date columns
                    if (
                        not recovered
                        and "date_trunc" in lower
                        and ("binder error" in lower or "candidate functions" in lower)
                    ):
                        recovery_sql = re.sub(
                            r"date_trunc\(\s*([^,]+),\s*([a-zA-Z_][a-zA-Z0-9_]*)\s*\)",
                            r"date_trunc(\1, TRY_CAST(\2 AS DATE))",
                            step.query_hint,
                            flags=re.IGNORECASE,
                        )
                        if recovery_sql != step.query_hint:
                            try:
                                df_fix = conn.execute(recovery_sql).fetchdf()
                                last_sql_df = df_fix
                                conn.register(f"step_{step.step_number}", df_fix)
                                conn.register("last_result", df_fix)
                                has_last_result = True
                                output = (
                                    "Recovered by casting date columns for date_trunc.\n"
                                    f"Original error: {msg}\n\n"
                                    + df_fix.to_string(index=False)
                                )
                                results.append(StepResult(
                                    step_number=step.step_number,
                                    tool_used=step.tool,
                                    output=output[:2000],
                                    success=True,
                                ))
                                recovered = True
                            except Exception:
                                recovered = False

                    # Recovery 2: reroute table reference to last_result for dependent steps
                    if (
                        not recovered
                        and has_last_result
                        and "referenced column" in lower
                        and "not found in from clause" in lower
                    ):
                        recovery_sql = re.sub(
                            r"\bfrom\s+(sales|employees)\b",
                            "FROM last_result",
                            step.query_hint,
                            flags=re.IGNORECASE,
                        )
                        if recovery_sql != step.query_hint:
                            try:
                                df_fix = conn.execute(recovery_sql).fetchdf()
                                last_sql_df = df_fix
                                conn.register(f"step_{step.step_number}", df_fix)
                                conn.register("last_result", df_fix)
                                has_last_result = True
                                output = (
                                    "Recovered by executing query against last_result.\n"
                                    f"Original error: {msg}\n\n"
                                    + df_fix.to_string(index=False)
                                )
                                results.append(StepResult(
                                    step_number=step.step_number,
                                    tool_used=step.tool,
                                    output=output[:2000],
                                    success=True,
                                ))
                                recovered = True
                            except Exception:
                                recovered = False

                if not recovered:
                    results.append(StepResult(
                        step_number=step.step_number,
                        tool_used=step.tool,
                        output=(
                            f"Failed: {e}\n"
                            "Hint: SQL steps must be self-contained or query step_<n>/last_result intermediates."
                        ),
                        success=False,
                    ))

        return results
    finally:
        conn.close()

# Execute the plan
print("Executing plan...")
step_results = execute_plan(plan, deps)

print(f"\nResults:")
for sr in step_results:
    status = "OK" if sr.success else "FAIL"
    print(f"  Step {sr.step_number} [{status}]: {sr.output[:150]}...")


Executing plan...
  Executing step 1: Inspect the sales table schema to confirm available columns ...
  Executing step 2: Calculate total profit (revenue - cost) for each unique prod...
  Executing step 3: Calculate aggregate statistics (total profit, total revenue,...

Results:
  Step 1 [OK]: Dataset: sales
Shape: 40 rows x 8 columns

Columns:
  - date (str)
    Unique values: 14, Nulls: 0
    Samples: 2024-01-05, 2024-01-12, 2024-01-19, 20...
  Step 2 [OK]:  product region  total_profit
Gadget Y   East       4898.04...
  Step 3 [OK]:  product region  total_profit  total_revenue  avg_unit_price  avg_other_profit  max_other_profit
Gadget Y   East       4898.04        12245.1         ...


In [6]:
# -----------------------------------------------------------------
# Step 3: The Synthesizer — turns step results into final answer
# -----------------------------------------------------------------

synthesizer = Agent(
    "openai:local-model",
    output_type=FinalAnalysis,
    system_prompt=(
        "You are a data analyst. Given the results of a multi-step analysis, "
        "synthesize a clear, complete answer. Include specific numbers. "
        "Be concise but thorough."
    ),
)

# Build context from step results
context = f"Original question: {plan.question}\n\nStep results:\n"
for sr in step_results:
    context += f"\nStep {sr.step_number} ({sr.tool_used}):\n{sr.output[:1000]}\n"

final = run_agent_with_limits(
    synthesizer,
    context,
    request_limit=40,
    tool_calls_limit=20,
)

print(f"Final Answer:\n{final.output.answer}\n")
print("Key findings:")
for f in final.output.key_findings:
    print(f"  - {f}")
print(f"\nConfidence: {final.output.confidence:.0%}")


Final Answer:
The most profitable product-region combination is **Gadget Y in the East region**, generating a total profit of **$4,898.04**.

This combination stands out for the following reasons:
1.  **Highest Absolute Profit:** It achieved a total profit of $4,898.04, which is significantly higher than the next best performing combination (which had a maximum profit of $4,799.04).
2.  **Strong Revenue Generation:** It contributed $12,245.10 in total revenue, indicating high sales volume or effective pricing strategies in that region.
3.  **Competitive Pricing:** The average unit price for this combination was $24.99, which is mid-range compared to the dataset's overall mean of $29.61, suggesting that Gadget Y in the East region drives profit through volume rather than just high unit margins.
4.  **Profit Margin Efficiency:** With a total profit of $4,898.04 against $12,245.10 in revenue, this combination maintains a healthy profit margin while outperforming other product-region pairi

### The Plan-Execute-Synthesize Pipeline

```
User Question
     |
     v
[PLANNER] → AnalysisPlan (structured steps)
     |
     v
[EXECUTOR] → StepResult[] (tool outputs)
     |
     v
[SYNTHESIZER] → FinalAnalysis (coherent answer)
```

Each stage is a separate agent with a focused job. This is a **pipeline architecture** —
simpler to debug, test, and improve than a single monolithic agent.

---

## Part 3: Production Controls — Guardrails on the Loop

An unconstrained agent loop is dangerous:
- It could run forever (infinite tool calls)
- It could blow your budget (hundreds of LLM calls)
- It could waste time on unproductive approaches

Production agents need **guardrails**.

### What guardrails to set and how to choose values

| Guardrail | Typical Values | How to Choose |
|-----------|---------------|---------------|
| `max_tool_calls` | 5-10 for simple, 15-20 for complex | Start low, increase based on eval results |
| `max_cost_usd` | $0.01-0.10 per query | Based on your margin (revenue per query - cost) |
| `timeout_seconds` | 10-30 per tool call | Depends on tool (SQL=5s, code=30s) |
| `retries` | 2-3 | Higher = more self-correction, but more cost |

### Budget-aware prompting

The most effective guardrail is **telling the LLM about its budget**.
When the system prompt says "You have 6 tool calls remaining", the LLM
naturally becomes more efficient — combining queries, skipping unnecessary inspections.

This works because LLMs understand resource constraints from training data.

In [7]:
# -----------------------------------------------------------------
# Guardrailed agent with iteration limits and cost tracking
# -----------------------------------------------------------------

MODEL_PRICING = {
    "gpt-4o-mini": (0.15, 0.60),  # per 1M tokens: (input, output)
    "gpt-4o": (2.50, 10.00),
    "claude-3-5-haiku-latest": (0.80, 4.00),
}


@dataclass
class GuardedDeps:
    tables: dict[str, pd.DataFrame] = field(default_factory=dict)
    data_dir: str = ""
    # Guardrails
    max_tool_calls: int = 8
    max_cost_usd: float = 0.05
    # Tracking
    tool_call_count: int = 0
    total_tokens: int = 0
    step_log: list[dict] = field(default_factory=list)

    def check_budget(self) -> None:
        """Raise if we've exceeded any limit."""
        if self.tool_call_count >= self.max_tool_calls:
            raise RuntimeError(
                f"Tool call limit reached ({self.max_tool_calls}). "
                f"Stopping to prevent runaway execution."
            )

    def log_step(self, tool: str, detail: str) -> None:
        self.tool_call_count += 1
        self.step_log.append({
            "step": self.tool_call_count,
            "tool": tool,
            "detail": detail[:200],
        })
        self.check_budget()


guarded_agent = Agent(
    "openai:local-model",
    deps_type=GuardedDeps,
    output_type=AnalysisResult,
    system_prompt=(
        "You are an efficient data analyst. Answer questions using the minimum "
        "number of tool calls needed.\n\n"
        "EFFICIENCY RULES:\n"
        "- Combine multiple aggregations into a single SQL query when possible\n"
        "- Don't inspect schema if the column names are clear from context\n"
        "- Stop as soon as you have enough data to answer confidently\n"
        "- You have a LIMITED budget of tool calls — use them wisely"
    ),
    retries=2,
)


@guarded_agent.system_prompt
def guarded_context(ctx: RunContext[GuardedDeps]) -> str:
    lines = ["\nAvailable tables:"]
    for name, df in ctx.deps.tables.items():
        cols = ", ".join(f"{c} ({df[c].dtype})" for c in df.columns)
        lines.append(f"  '{name}': {len(df)} rows | {cols}")
    remaining = ctx.deps.max_tool_calls - ctx.deps.tool_call_count
    lines.append(f"\nBudget: {remaining} tool calls remaining out of {ctx.deps.max_tool_calls}")
    return "\n".join(lines)


@guarded_agent.tool
def guarded_sql(ctx: RunContext[GuardedDeps], sql: str) -> str:
    """Run SQL query. Budget-tracked."""
    ctx.deps.log_step("sql", sql)
    conn = duckdb.connect()
    try:
        for name, df in ctx.deps.tables.items():
            conn.register(name, df)
        result = conn.execute(sql).fetchdf()
        if len(result) > 30:
            return f"First 30 of {len(result)} rows:\n{result.head(30).to_string()}"
        return result.to_string()
    except Exception as e:
        raise ModelRetry(f"SQL error: {e}")
    finally:
        conn.close()


@guarded_agent.tool
def guarded_inspect(ctx: RunContext[GuardedDeps], table_name: str) -> str:
    """Inspect table schema. Budget-tracked."""
    ctx.deps.log_step("inspect", table_name)
    if table_name not in ctx.deps.tables:
        return f"Not found. Available: {list(ctx.deps.tables.keys())}"
    return inspect_schema(ctx.deps.tables[table_name], table_name)


# Run with guardrails
guarded_deps = GuardedDeps(
    tables={"sales": sales_df, "employees": employees_df},
    data_dir=DATA_DIR,
    max_tool_calls=6,
)

result = run_agent_with_limits(
    guarded_agent,
    "Give me a full business overview: total revenue, top product, "
    "best region, average profit margin, and headcount by department.",
    deps=guarded_deps,
    request_limit=80,
    tool_calls_limit=40,
)

print(f"Answer:\n{result.output.answer}\n")
print(f"Tool calls used: {guarded_deps.tool_call_count}/{guarded_deps.max_tool_calls}")
print(f"\nExecution trace:")
for step in guarded_deps.step_log:
    print(f"  {step['step']}. [{step['tool']}] {step['detail'][:80]}")


Answer:
**Business Overview:**

• **Total Revenue:** $147,695.10
• **Top Product:** Widget A ($36,887.70 revenue)
• **Best Region:** East ($39,311.20 revenue)
• **Average Profit Margin:** 40.0%

**Headcount by Department:**
- Engineering: 10 employees
- Marketing: 4 employees
- Sales: 4 employees
- HR: 2 employees

**Total Headcount:** 20 employees across 4 departments

Tool calls used: 4/6

Execution trace:
  1. [sql] SELECT 
    SUM(revenue) as total_revenue,
    AVG((revenue - cost) / revenue * 
  2. [sql] SELECT product, SUM(revenue) as revenue FROM sales GROUP BY product ORDER BY rev
  3. [sql] SELECT region, SUM(revenue) as revenue FROM sales GROUP BY region ORDER BY reven
  4. [sql] SELECT department, COUNT(employee_id) as headcount FROM employees GROUP BY depar


### Key guardrail patterns

| Guardrail | What it prevents | Implementation |
|-----------|-----------------|----------------|
| Max tool calls | Infinite loops | Counter in deps, check before each tool |
| Cost budget | Runaway spending | Token tracking, stop at threshold |
| Timeout per step | Hung tool calls | `timeout_seconds` on code execution |
| Output truncation | Context overflow | Limit tool output to N chars |
| Retry limit | Infinite retry loops | `retries=N` on agent |

---

## Part 4: Comparing Strategies — Which Works Best?

Let's run the same complex question through both approaches and compare.

In [8]:
COMPLEX_QUESTION = (
    "Identify the top 3 most profitable product-region pairs. "
    "For each, show total revenue, total profit, and profit margin. "
    "Are any of these trending up or down over time?"
)

# --- Approach 1: Implicit ReAct (single agent with tools) ---
deps1 = AnalystDeps(
    tables={"sales": sales_df, "employees": employees_df},
    data_dir=DATA_DIR,
)
start = time.time()
result1 = run_agent_with_limits(
    react_agent,
    COMPLEX_QUESTION,
    deps=deps1,
    request_limit=120,
    tool_calls_limit=80,
)
time1 = time.time() - start
usage1 = result1.usage()

print("=== IMPLICIT REACT ===")
print(f"Answer: {result1.output.answer[:300]}...")
print(f"Steps: {result1.output.steps_taken} | Tokens: {usage1.input_tokens + usage1.output_tokens}")
print(f"Time: {time1:.1f}s | Confidence: {result1.output.confidence:.0%}")

# --- Approach 2: Plan-Execute-Synthesize ---
start = time.time()
plan_r = run_agent_with_limits(
    planner,
    COMPLEX_QUESTION,
    deps=deps1,
    request_limit=60,
    tool_calls_limit=30,
)
step_results = execute_plan(plan_r.output, deps1)

context = f"Question: {plan_r.output.question}\n\nResults:\n"
for sr in step_results:
    context += f"Step {sr.step_number} ({sr.tool_used}): {sr.output[:800]}\n\n"
final = run_agent_with_limits(
    synthesizer,
    context,
    request_limit=40,
    tool_calls_limit=20,
)
time2 = time.time() - start

print(f"\n=== PLAN-EXECUTE-SYNTHESIZE ===")
print(f"Answer: {final.output.answer[:300]}...")
print(f"Plan steps: {len(plan_r.output.steps)} | Succeeded: {sum(1 for s in step_results if s.success)}")
print(f"Time: {time2:.1f}s | Confidence: {final.output.confidence:.0%}")


=== IMPLICIT REACT ===
Answer: The top 3 most profitable product-region pairs are:

1. **Gadget Y - East**: $12,245.10 revenue, $4,898.04 profit, 40% margin - Trend: FLAT/STABLE
2. **Widget B - North**: $11,997.60 revenue, $4,799.04 profit, 40% margin - Trend: UP
3. **Widget A - East**: $10,796.40 revenue, $4,318.56 profit, 40% m...
Steps: 3 | Tokens: 18720
Time: 84.8s | Confidence: 95%
  Executing step 1: Calculate total revenue, cost, and profit for each product-r...
  Executing step 2: For each of the top 3 product-region pairs identified in ste...

=== PLAN-EXECUTE-SYNTHESIZE ===
Answer: The top 3 most profitable product-region pairs are:

1. **Gadget Y - East**: Total Revenue $12,245.10, Total Profit $4,898.04, Profit Margin 40.0%. Trend: Fluctuating (Jan: $3,998.40 → Feb: $4,373.25 → Mar: $3,873.45).
2. **Widget B - North**: Total Revenue $11,997.60, Total Profit $4,799.04, Profit...
Plan steps: 2 | Succeeded: 2
Time: 36.7s | Confidence: 95%


### When to use which approach — Decision guide

```
User question arrives
       |
       v
[Complexity classifier]  ← From Lesson 1
       |
       +-- "simple" → Implicit ReAct (single agent, 1-3 tool calls)
       |                 Why: minimal overhead, fast, cheap
       |
       +-- "medium" → Implicit ReAct with guardrails (max 6-8 tool calls)
       |                 Why: flexibility with safety rails
       |
       +-- "complex" → Plan-Execute-Synthesize
                         Why: auditable, debuggable, human-reviewable
```

| Question Type | Example | Best Approach | Why |
|---------------|---------|---------------|-----|
| Direct lookup | "Total revenue?" | Implicit, 1 tool call | Overkill to plan |
| Single aggregation | "Revenue by region?" | Implicit, 2-3 tool calls | Quick enough |
| Cross-table analysis | "Revenue vs headcount?" | Implicit with guardrails | May need joins |
| Multi-metric comparison | "Compare categories across 5 metrics" | Plan-Execute-Synthesize | Need coordinated steps |
| Trend + comparison | "Top products trending up vs down?" | Plan-Execute-Synthesize | Multi-step reasoning |

---

## Part 5: Critique + Verification — Industry Pattern

Leading agent teams use a **Generate → Critique → Revise** loop for higher-stakes tasks.

This pattern gives you two protections:
1. **Deterministic verifier checks** (fast, cheap, objective)
2. **LLM critic pass** (semantic quality and completeness checks)

In practice, run critique selectively: complex queries, low confidence answers, or failed checks.

In [9]:
# -----------------------------------------------------------------
# Critic + verifier components
# -----------------------------------------------------------------

class CritiqueReport(BaseModel):
    verdict: str = Field(default="accept", description="accept or revise")
    issues: list[str] = Field(default_factory=list)
    missing_checks: list[str] = Field(default_factory=list)
    improvement_actions: list[str] = Field(default_factory=list)
    confidence: float = Field(default=0.7)

    @field_validator("verdict", mode="before")
    @classmethod
    def coerce_verdict(cls, v):
        value = str(v).strip().lower() if v is not None else "accept"
        return value if value in {"accept", "revise"} else "accept"

    @field_validator("issues", "missing_checks", "improvement_actions", mode="before")
    @classmethod
    def coerce_list_fields(cls, v):
        if v is None:
            return []
        if isinstance(v, str):
            items = [line.strip("- *•\t ") for line in v.splitlines() if line.strip()]
            return items or [v.strip()]
        if isinstance(v, list):
            return [str(item) for item in v]
        return [str(v)]

    @field_validator("confidence", mode="before")
    @classmethod
    def coerce_confidence(cls, v):
        if isinstance(v, str):
            raw = v.strip().replace("%", "")
            try:
                v = float(raw)
            except ValueError:
                return 0.7
        try:
            conf = float(v)
        except (TypeError, ValueError):
            return 0.7
        if conf > 1.0 and conf <= 100.0:
            conf = conf / 100.0
        return max(0.0, min(conf, 1.0))


class VerificationReport(BaseModel):
    passed: bool = True
    checks: list[str] = Field(default_factory=list)
    failed_checks: list[str] = Field(default_factory=list)


def verify_answer_coverage(answer: str, step_results: list[StepResult]) -> VerificationReport:
    checks: list[str] = []
    failed: list[str] = []

    successful_outputs = "\n".join(sr.output.lower() for sr in step_results if sr.success)
    if not successful_outputs:
        failed.append("No successful execution outputs available for evidence grounding.")
    else:
        checks.append("Execution produced at least one successful step.")

    failed_steps = [sr for sr in step_results if not sr.success]
    if failed_steps:
        answer_lower = answer.lower()
        acknowledges_limits = any(
            phrase in answer_lower
            for phrase in ["could not", "unable", "failed", "insufficient", "cannot determine"]
        )
        if acknowledges_limits:
            checks.append("Execution failures were transparently acknowledged in the answer.")
        else:
            failed.append("One or more execution steps failed without being acknowledged in the answer.")
    else:
        checks.append("All execution steps succeeded.")

    # Compare numeric claims with tolerance rather than raw string equality
    claim_pattern = r"(?:\$?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?%?)"
    numeric_claims = re.findall(claim_pattern, answer)
    evidence_numbers_raw = re.findall(claim_pattern, successful_outputs)

    def _parse_number(num_text: str):
        t = num_text.strip().replace("$", "").replace(",", "")
        is_percent = t.endswith("%")
        t = t[:-1] if is_percent else t
        try:
            return float(t), is_percent
        except ValueError:
            return None, is_percent

    evidence_numbers: list[float] = []
    for raw in evidence_numbers_raw:
        val, _ = _parse_number(raw)
        if val is not None:
            evidence_numbers.append(val)

    unmatched_claims: list[str] = []
    for claim in numeric_claims[:20]:
        c, c_is_percent = _parse_number(claim)
        if c is None:
            continue
        # Skip tiny integers (often list indices/ranks) to reduce false alarms.
        if c.is_integer() and abs(c) < 10 and "%" not in claim and "$" not in claim:
            continue

        # Allow percent-vs-ratio equivalence: 40% <-> 0.4
        candidate_values = [c]
        if c_is_percent:
            candidate_values.append(c / 100.0)
        elif c <= 1.0:
            candidate_values.append(c * 100.0)

        matched = any(
            any(abs(cv - ev) <= max(0.02, abs(cv) * 0.0005) for ev in evidence_numbers)
            for cv in candidate_values
        )
        if not matched:
            unmatched_claims.append(claim)

    if unmatched_claims:
        failed.append("Numeric claims not grounded in step evidence: " + ", ".join(unmatched_claims[:6]))
    else:
        checks.append("Numeric claims are grounded in execution evidence.")

    return VerificationReport(passed=len(failed) == 0, checks=checks, failed_checks=failed)


critic_agent = Agent(
    "openai:local-model",
    output_type=CritiqueReport,
    system_prompt=(
        "You are a strict reviewer for data-analysis agent outputs.\n"
        "Use only provided execution evidence and verifier results.\n"
        "If evidence is insufficient, set verdict='revise'.\n"
        "Focus on: correctness, completeness, and unsupported claims."
    ),
)

reviser_agent = Agent(
    "openai:local-model",
    output_type=FinalAnalysis,
    system_prompt=(
        "You revise analysis answers using reviewer feedback and execution evidence.\n"
        "Do not invent numbers. Keep the answer concise and factual."
    ),
)

In [10]:
def run_critique_cycle(question: str, draft: FinalAnalysis, step_results: list[StepResult]):
    verifier = verify_answer_coverage(draft.answer, step_results)

    evidence = "\n\n".join(
        f"Step {sr.step_number} ({sr.tool_used}, success={sr.success}):\n{sr.output[:1200]}"
        for sr in step_results
    )

    critique_prompt = (
        f"Question:\n{question}\n\n"
        f"Draft answer:\n{draft.answer}\n\n"
        f"Key findings: {draft.key_findings}\n\n"
        f"Verifier checks passed: {verifier.passed}\n"
        f"Verifier check list: {verifier.checks}\n"
        f"Verifier failures: {verifier.failed_checks}\n\n"
        f"Execution evidence:\n{evidence}\n"
    )

    critique = run_agent_with_limits(
        critic_agent,
        critique_prompt,
        request_limit=40,
        tool_calls_limit=5,
    ).output

    revised = False
    final_output = draft

    if critique.verdict == "revise" or not verifier.passed:
        revise_prompt = (
            f"Original question:\n{question}\n\n"
            f"Current answer:\n{draft.answer}\n\n"
            f"Reviewer issues: {critique.issues}\n"
            f"Missing checks: {critique.missing_checks}\n"
            f"Improvement actions: {critique.improvement_actions}\n"
            f"Verifier failures: {verifier.failed_checks}\n\n"
            f"Execution evidence:\n{evidence}\n"
        )
        final_output = run_agent_with_limits(
            reviser_agent,
            revise_prompt,
            request_limit=50,
            tool_calls_limit=5,
        ).output
        revised = True

    return {
        "verifier": verifier,
        "critique": critique,
        "final": final_output,
        "revised": revised,
    }

critique_run = run_critique_cycle(
    COMPLEX_QUESTION,
    final.output,
    step_results,
)

print("=== CRITIQUE + VERIFY ===")
print(f"Verifier passed: {critique_run['verifier'].passed}")
print(f"Verifier failures: {critique_run['verifier'].failed_checks}")
print(f"Critic verdict: {critique_run['critique'].verdict}")
print(f"Critic issues: {critique_run['critique'].issues}")
print(f"Revised: {critique_run['revised']}")
print("\nFinal answer after critique cycle:")
print(critique_run['final'].answer)
print("\nTool path used as evidence:")
for sr in step_results:
    status = "OK" if sr.success else "FAIL"
    print(f"  Step {sr.step_number} [{status}] via {sr.tool_used}")

=== CRITIQUE + VERIFY ===
Verifier passed: True
Verifier failures: []
Critic verdict: revise
Critic issues: ["Monthly trend values incorrectly show revenue instead of profit - all three pairs display monthly_revenue figures when discussing 'monthly_profit' trends", "The claim 'Only Widget A - East demonstrates an upward trend' is misleading - Widget B - North also shows positive net profit change (+$199.96 vs +$419.86 for Widget A - East)", 'Trend analysis uses wrong metric (revenue) which could lead to different conclusions than profit-based trend analysis']
Revised: True

Final answer after critique cycle:
The top 3 most profitable product-region pairs are:

1. **Gadget Y - East**: Total Revenue $12,245.10, Total Profit $4,898.04, Profit Margin 40.0%. Trend: Fluctuating (Jan: $1,599.36 → Feb: $1,749.30 → Mar: $1,549.38).
2. **Widget B - North**: Total Revenue $11,997.60, Total Profit $4,799.04, Profit Margin 40.0%. Trend: Upward (Jan: $1,399.72 → Feb: $1,799.64 → Mar: $1,599.68). Net

### Critic pattern takeaways

- Deterministic checks catch objective failures quickly (missing numbers, failed steps).
- The critic handles semantic quality (gaps, weak explanation, unsupported conclusions).
- The reviser is only invoked when needed, which controls latency and cost.

This is the practical pattern used in production agents: **selective critique, not universal critique**.
Run it on high-impact or low-confidence outputs.

## Summary

| Concept | What | Why |
|---------|------|-----|
| ReAct loop | Think-Act-Observe cycle | Systematic multi-step reasoning |
| Implicit ReAct | PydanticAI's native tool loop | Simple, flexible |
| Explicit planning | Planner agent outputs structured plan | Debuggable, auditable |
| Plan-Execute-Synthesize | 3-stage pipeline | Separation of concerns |
| Guardrails | Max steps, cost budget, timeouts | Prevent runaway agents |
| Step logging | Track every tool call | Debugging, cost analysis |
| Critique + Verify | Deterministic checks + LLM reviewer pass | Higher accuracy on hard queries |

### Production patterns
1. **Always set max iterations** — Uncapped loops are production incidents
2. **Log every step** — You need full traces for debugging
3. **Budget-aware agents** — Tell the LLM its remaining budget
4. **Separate planning from execution** — Easier to test each stage
5. **Complexity routing** — Simple questions skip planning overhead
6. **Selective critique pass** — Run verifier + critic for high-impact answers

**Next: Lesson 5 — Memory (agent remembers context across turns)**


---
## Exercises

1. **Human-in-the-loop** — Show the plan to the user and let them approve/modify before execution.
2. **Adaptive replanning** — If a step fails, have the planner create a new plan using what succeeded.
3. **Parallel execution** — Execute independent steps concurrently with `asyncio.gather`.
4. **Cost estimator** — Before executing, estimate the cost of the plan based on expected token usage.